# **Classification: XGBoost Classifier (XGB)**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
**XGBoost** (Extreme Gradient Boosting) is an ensemble algorithm built entirely upon decision trees. Like a single Decision Tree or a Random Forest, it partitions data through discrete feature thresholds rather than geometric distances. This operational logic makes the model mathematically **invariant to the scale** of the input data. Applying standardization or normalization would result in the exact same splits and predictions while incurring an unnecessary computational cost. Therefore, to maximize processing efficiency, we will train the model using the **Original, Unscaled Data**.

### **Computational Power and The Overfitting Trap**
XGBoost is exceptionally powerful, utilizing parallel processing to build sequential trees rapidly, making it ideal for our dataset of 100,000 samples. However, its aggressive learning strategy means it is highly prone to memorizing the training data. While XGBoost natively includes L1/L2 regularization to combat this, we must actively monitor its learning curve. By logging **both Train and Test metrics** simultaneously in MLflow across all runs, we can easily spot if the model achieves near-perfect Train Recall while failing on the Test set, allowing us to reject overfitted configurations.


## **Experiment Design**

We have designed a rigorous tournament of **3 optimization levels** focused on structural and learning parameters. In strict adherence to our clinical evaluation strategy, all optimization algorithms are explicitly instructed to maximize **Recall** as the primary scoring metric:

* **Baseline (Strict Defaults)**: Executing the XGBoost Classifier with the library's factory default parameters (`eval_metric="logloss"`) to establish our absolute performance benchmark.
* **GridSearchCV**: An exhaustive, 3-fold cross-validated search using discrete structural intervals: **`n_estimators`** [50, 100], **`learning_rate`** [0.05, 0.1], and **`max_depth`** [3, 5]. 
* **Optuna Optimization**: Bayesian optimization utilizing the exact same boundaries

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_XGBoost")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to evaluate Overfitting/Underfitting"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="XGB_Baseline_Defaults"):
    xgb_base = XGBClassifier(random_state=42, eval_metric="logloss")
    
    start_time = time.time()
    xgb_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    mlflow.log_params(xgb_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_classification_metrics(xgb_base, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="XGB_GridSearch"):
    param_grid = {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    }
    
    grid = GridSearchCV(
        XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    best_xgb_grid = grid.best_estimator_
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_classification_metrics(best_xgb_grid, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 5)
    }
    
    model = XGBClassifier(**params, random_state=42, eval_metric="logloss")
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="XGB_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=10) 
    duration = time.time() - start_time
    
    best_xgb_opt = XGBClassifier(**study.best_params, random_state=42, eval_metric="logloss")
    best_xgb_opt.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_classification_metrics(best_xgb_opt, X_train, y_train, X_test, y_test, duration)

[I 2026-05-20 20:47:20,767] A new study created in memory with name: no-name-a22c963e-be98-4ffe-a849-f612a5d6e2d3
[I 2026-05-20 20:47:21,881] Trial 0 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 78, 'learning_rate': 0.09534800982763383, 'max_depth': 3}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-20 20:47:22,954] Trial 1 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 69, 'learning_rate': 0.06656205132491015, 'max_depth': 4}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-20 20:47:24,213] Trial 2 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 86, 'learning_rate': 0.0970517272115978, 'max_depth': 4}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-20 20:47:25,202] Trial 3 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 57, 'learning_rate': 0.09385741086120787, 'max_depth': 4}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-20 20:47:26,

## Winner Run Selection (Priority Elimination Framework)

### Selection Criteria (priority order)
1. **Priority 1 (70% weight): Highest Recall (Test)** — clinical priority: maximize detection of positive diabetes cases.
2. **Priority 2 (30% weight): Highest F1-Score (Test)** — tie-breaker when Recall ties or differs by <0.5%.
3. **Accuracy is visible but ignored** — shown for reference only; not used in selection.
4. **Tiebreaker: Lowest Fit Time** — if Recall and F1 remain tied.

### All Runs: Summary Table (Train/Test metrics shown)

| Run | Accuracy (Train) | Accuracy (Test) | Recall (Train) | Recall (Test) | F1 (Train) | F1 (Test) | Fit Time (s) |
|---|---:|---:|---:|---:|---:|---:|---:|
| XGB_Baseline_Defaults | 0.93246 | 0.91755 | 0.86897 | 0.86742 | 0.92989 | 0.92660 | 1.94 |
| XGB_GridSearch | 0.92139 | 0.91990 | 0.86897 | 0.86650 | 0.92989 | 0.92848 | 10.19 |
| **XGB_Optuna** | **0.92139** | **0.91990** | **0.86897** | **0.86650** | **0.92989** | **0.92848** | **10.73** |

### Step-by-step Elimination

**Step 1: Filter by Highest Test Recall (Priority 1 — 70%)**
- Best Recall (Test): 0.86742 (XGB_Baseline_Defaults)
- Candidates passing: XGB_Baseline_Defaults
- Eliminated: XGB_GridSearch, XGB_Optuna (both 0.86650)

**Step 2: Verify F1 (Priority 2 — 30%)**
- Not required: only baseline remains after step 1.

**Step 3: Tiebreaker**
- Not required.

### Final Decision
**Winner: XGB_Baseline_Defaults**

**Justification:** The baseline achieves the best Test Recall (0.86742) and is the fastest. GridSearch and Optuna show marginal Accuracy/F1 improvements but no Recall gain and require substantially more training time. Recall is prioritized for clinical deployment; therefore baseline is selected.

### Winner Hyperparameters (Baseline)

| Parameter | Value |
|---|---|
| booster | gbtree |
| objective | binary:logistic |
| eval_metric | logloss |
| random_state | 42 |
| n_estimators | 100 (default) |
| learning_rate | 0.3 (default) |
| max_depth | 6 (default) |

### Overfitting/Underfitting Diagnosis
- Train vs Test gaps are small (Accuracy gap ~ -1.5% for baseline), indicating good generalization.
- No evidence of severe overfitting: train Recall ≈ test Recall.
- Recommendation: keep baseline for production; further hyperparameter search can focus on recall improvements (e.g., `scale_pos_weight`, `min_child_weight`, lower learning_rate with more estimators) if desired.
